In [ ]:
!pip install --quiet boto3

In [ ]:
from google.colab import userdata
import os

# Get AWS credentials from Colab Secrets
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_SESSION_TOKEN'] = userdata.get('AWS_SESSION_TOKEN')

print("AWS credentials set as environment variables.")

In [ ]:
os.environ['AWS_ACCESS_KEY_ID']

In [ ]:
os.environ['AWS_SECRET_ACCESS_KEY']

In [ ]:
os.environ['AWS_SESSION_TOKEN']

In [ ]:
import boto3

endpoint_name = "linear-learner10Millones-endpoint"

runtime = boto3.client("sagemaker-runtime", region_name="us-east-1")

# Ejemplo: una fila con 10 features
payload = "0.49671415,-0.13826430,0.64768854,1.52302986,-0.23415337,-0.23413696,1.57921282,0.76743473,-0.46947439,0.54256004"

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=payload
)

print(response["Body"].read().decode("utf-8"))


In [ ]:

import gradio as gr
import boto3

endpoint_name = "linear-learner10Millones-endpoint" # Make sure this matches your endpoint name
region_name = "us-east-1" # Make sure this matches your endpoint region

runtime = boto3.client("sagemaker-runtime", region_name=region_name)

def predict(input_data):
    try:
        # Assuming input_data is a string of comma-separated values
        payload = input_data
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType="text/csv",
            Body=payload
        )
        result = response["Body"].read().decode("utf-8")
        return result
    except Exception as e:
        return f"Error: {e}"

iface = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="Enter comma-separated features (e.g., 0.45,-0.67,0.89,1.0,-0.2,0.33,0.12,0.9,-0.56,1.22)"),
    outputs="text",
    title="SageMaker Linear Learner Endpoint Prediction",
    description="Enter a comma-separated string of feature values to get a prediction from the SageMaker endpoint."
)

iface.launch()